In [64]:
from tensorflow.keras.models import load_model
import pandas as pd
import numpy as np
import pickle

In [65]:
## Load all my preprocessor and model

model = load_model('model.h5')

with open('label_encoder_gender.pkl','rb') as file:
    label_encoder_gender = pickle.load(file)

with open('one_hot_encoder_geo.pkl','rb') as file:
    one_hot_encoder_geo = pickle.load(file)

with open('scaler.pkl','rb') as file:
    scaler = pickle.load(file)

In [66]:
# example input data

input_data = {
    'CreditScore' : 600,
    'Geography' : 'France',
    'Gender' : 'Male',
    'Age' : 40,
    'Tenure': 3,
    'Balance': 60000,
    'NumOfProducts': 3,
    'HasCrCard': 1,
    'IsActiveMember': 1,
    'EstimatedSalary': 51000
}

In [67]:
input_data_df = pd.DataFrame(data=[input_data])
input_data_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,Male,40,3,60000,3,1,1,51000


In [68]:
input_data_df['Gender'] = label_encoder_gender.transform(input_data_df['Gender'])
input_data_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,1,40,3,60000,3,1,1,51000


In [69]:
geo_encoded = one_hot_encoder_geo.transform(input_data_df[['Geography']])
geo_encoded_df = pd.DataFrame(geo_encoded,columns=one_hot_encoder_geo.get_feature_names_out(['Geography']))#['Geography_France','Geography_Germany','Geography_Spain'])
geo_encoded

array([[1., 0., 0.]])

In [70]:
input_data_df = pd.concat([input_data_df.drop(['Geography'],axis=1).reset_index(drop=True),geo_encoded_df],axis=1)
input_data_df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,1,40,3,60000,3,1,1,51000,1.0,0.0,0.0


In [71]:
input_data_df = scaler.transform(input_data_df)
input_data_df

array([[-0.53598516,  0.91324755,  0.10479359, -0.69539349, -0.25781119,
         2.53355998,  0.64920267,  0.97481699, -0.85944554,  1.00150113,
        -0.57946723, -0.57638802]])

In [78]:
# Predict

pred = model.predict(input_data_df)
pred = pred[0][0]
pred

1/1 [==============================] - 0s 45ms/step


0.8376225

In [79]:
if pred>0.5:
    print('Customer likely to churn')
else:
    print('Customer not likely to churn')

Customer likely to churn


In [82]:
one_hot_encoder_geo.categories_[0]

array(['France', 'Germany', 'Spain'], dtype=object)

In [85]:
label_encoder_gender.classes_

array(['Female', 'Male'], dtype=object)